### Matching

Match extracted resume skills with job description skills
* Calculate dictionary score
* Calculate jaccard score 

In [13]:
# Dictionary score
from importlib import reload
import sys
from pathlib import Path
import json
import pandas as pd

# Add ../src to import path
src_path = (Path.cwd().parent / "src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

reload(sys.modules["matching"])

from matching import compute_dictionary_pairs_and_topk, dumps_skill_columns
from matching import to_skill_list

processed_dir = "../data/processed"

In [14]:
# Use ranked extracted files if present, else fall back to raw files
resume_ranked_path = f"{processed_dir}/resume_skills_extracted.csv"
job_ranked_path = f"{processed_dir}/job_skills_extracted.csv"


resume_out = pd.read_csv(resume_ranked_path)
job_out = pd.read_csv(job_ranked_path)
print("Loaded ranked extracted files")


# Parse JSON-string skill columns back into lists if needed
for df in (resume_out, job_out):
    for col in ["extracted_skills", "extracted_skills_ranked"]:
        if col in df.columns:
            df[col] = df[col].apply(to_skill_list)

# Use ranked skills if present; otherwise fall back to raw extracted skills
resume_skill_col = "extracted_skills_ranked" if "extracted_skills_ranked" in resume_out.columns else "extracted_skills"
job_skill_col = "extracted_skills_ranked" if "extracted_skills_ranked" in job_out.columns else "extracted_skills"

# Compute dictionary pairs and top-K matches
# Each (resume, job) pair is scored based on skill overlap, and the top-K jobs are identified for each resume.
# 712 resumes x 3000 jobs = 2.136 million pairs
dictionary_pairs_df, dictionary_topk_df = compute_dictionary_pairs_and_topk(
    resume_df=resume_out,
    job_df=job_out,
    resume_skill_col=resume_skill_col,
    job_skill_col=job_skill_col,
    top_k=20,
    dictionary_weight=0.7,
    title_weight=0.3,
)

print("Pair rows:", dictionary_pairs_df.shape)
print("Top-K rows:", dictionary_topk_df.shape)
display(dictionary_topk_df.head(10))


Loaded ranked extracted files
Pair rows: (2136000, 16)
Top-K rows: (14240, 17)


,resume_id,resume_idx,resume_category,job_idx,job_link,job_title,company,job_location,dictionary_score,job_coverage,skill_jaccard,title_similarity,final_score,matched_skills,missing_skills,extra_skills,rank
0,20345168,0,ACCOUNTANT,1759,https://www.linkedin.com/jobs/view/accounting-...,Accounting Manager,MDC Partners,"Los Angeles, CA",23.00,0.3023,0.0613,0.85,41.60,"[analysis, billing, excel, outlook, payable, p...","[account payable, account receivable, accruals...","[accountant, accounting duties, accounting ser...",1
1,20345168,0,ACCOUNTANT,480,https://www.linkedin.com/jobs/view/accounts-re...,Accounts Receivable Supervisor/Manager,Robert Half,"Midland, TX",22.33,0.3000,0.0443,0.85,41.13,"[analysis, annual, customer, customer service,...","[accounting, administration, business, busines...","[accountant, accounting duties, accounting ser...",2
2,20345168,0,ACCOUNTANT,1188,https://www.linkedin.com/jobs/view/property-ac...,Property Accounting Supervisor,LHH,"Greensboro, NC",21.36,0.2941,0.0258,0.85,40.45,"[bank, excel, information, ledger, reconciliat...","[accounting, bank reconciliation, bs in accoun...","[accountant, accounting duties, accounting ser...",3
3,20345168,0,ACCOUNTANT,280,https://www.linkedin.com/jobs/view/staff-accou...,Staff Accountant,Willamette Falls Paper Company,"West Linn, OR",17.12,0.2188,0.0603,0.90,38.98,"[bank, customer, customer service, excel, inde...","[accounting principles, accounting procedures,...","[accountant, accounting duties, accounting ser...",4
4,20345168,0,ACCOUNTANT,698,https://www.linkedin.com/jobs/view/sr-staff-ac...,Sr. Staff Accountant,Robert Half,"Avondale, PA",16.27,0.2162,0.0379,0.90,38.39,"[analysis, excel, inventory, ledger, manufactu...","[accounting or finance, accounting reconciliat...","[accountant, accounting duties, accounting ser...",5
5,20345168,0,ACCOUNTANT,1195,https://www.linkedin.com/jobs/view/accountant-...,Accountant,"Coolray Heating, Cooling, Plumbing, Electrical","Birmingham, AL",11.15,0.1458,0.0314,1.00,37.80,"[analysis, excel, ledger, policies, quarterly,...","[accounting policies, accounting procedures, a...","[accountant, accounting duties, accounting ser...",6
6,20345168,0,ACCOUNTANT,1969,https://www.linkedin.com/jobs/view/fund-accoun...,"Fund Accountant - Fund Accounting, Finance, CPA",CyberCoders,"North Palm Beach, FL",15.36,0.2069,0.0293,0.90,37.75,"[analysis, excel, fund, fund accounting, micro...","[accounting, activity, ad hoc reporting, calls...","[accountant, accounting duties, accounting ser...",7
7,20345168,0,ACCOUNTANT,1445,https://www.linkedin.com/jobs/view/senior-acco...,Senior Accountant Tax,Paragon Accountants,"San Diego, CA",15.20,0.1964,0.0485,0.90,37.64,"[business management, deadlines, excel, financ...","[accounting, advanced, advanced excel, analyti...","[accountant, accounting duties, accounting ser...",8
8,20345168,0,ACCOUNTANT,2677,https://ca.linkedin.com/jobs/view/accountant-a...,Accountant,Kassen Recruitment,"Markham, Ontario, Canada",10.73,0.1400,0.0311,1.00,37.51,"[construction, excel, financial statement prep...","[accounting, accounting experience, adherence,...","[accountant, accounting duties, accounting ser...",9
9,20345168,0,ACCOUNTANT,2568,https://www.linkedin.com/jobs/view/sr-accounta...,Sr. Accountant,Robert Half,"Hemet, CA",14.77,0.1951,0.0372,0.90,37.34,"[analysis, industry, ledger, manufacturing, or...","[accounting, attention, attention to detail, a...","[accountant, accounting duties, accounting ser...",10


In [15]:
# Save outputs for downstream fusion
pairs_save = dumps_skill_columns(dictionary_pairs_df, ["matched_skills", "missing_skills"])
topk_save = dumps_skill_columns(dictionary_topk_df, ["matched_skills", "missing_skills"])

pairs_path = f"{processed_dir}/dictionary_pair_scores.csv"
topk_path = f"{processed_dir}/dictionary_topk_per_resume.csv"
pairs_save.to_csv(pairs_path, index=False)
topk_save.to_csv(topk_path, index=False)

print("Saved:")
print(pairs_path)
print(topk_path)

Saved:
../data/processed/dictionary_pair_scores.csv
../data/processed/dictionary_topk_per_resume.csv
